In [ ]:
import sys
# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [ ]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig
from src.qa_pipeline.knowledge_retriever.MixturedTripletsRetriever import MixturedGraphSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

from src.utils import Logger
from src.utils import Logger, ReaderMetrics

#### 1. Загружем датасет с текстами, на основе которого будет построен граф знаний

In [ ]:
# Loading dataset to build graph and other structures 
# TODO

DATASET_PATH = ...
data = ...

#### 2. Задаём конфигурацию графа знаний

In [ ]:
# in-memory storage
GRAPH_STORAGE_CONFIG = GraphDriverConfig(db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)
KV_STORAGE_CONFIG = KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)

In [ ]:
#
LANGUAGE = 'auto' # 'ru' , 'en', 'auto

#
RETRIEVER_NAME = 'astar' # 'astar', 'bfs'
RETRIEVER_HYPERP = AStarGraphSearchConfig(max_depth=10, max_passed_nodes=500) # AStarGraphSearchConfig, BFSSearchConfig 

# embedder hyperp
DEVICE = 'cuda'
EMBEDDER_MODEL_PATH = '../../models/intfloat/multilingual-e5-small'

# vector dbs hyperp
NODES_DB_PATH = '../../data/graph_structures/vectorized_nodes/testing'
TRIPLETS_DB_PATH = '../../data/graph_structures/vectorized_triplets/testing'
NEED_TO_CLEAR = True

In [ ]:
inmemory_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GRAPH_STORAGE_CONFIG),
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=NODES_DB_PATH, db_name='vectorized_nodes', need_to_clear=NEED_TO_CLEAR)),
        tripletsdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=TRIPLETS_DB_PATH, db_name='vectorized_triplets', need_to_clear=NEED_TO_CLEAR)),
        embedder_config=EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH, device=DEVICE)),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(lang=LANGUAGE),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method=RETRIEVER_NAME,retriever_config=RETRIEVER_HYPERP,
            cache_config=KV_STORAGE_CONFIG),
        answer_generator_config=QALLMGeneratorConfig(lang=LANGUAGE)),
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(lang=LANGUAGE),
        updator_config=LLMUpdatorConfig(lang=LANGUAGE)),
    log=Logger('log/main'))

#### 3. Инициализируем граф знаний

In [ ]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

In [ ]:
# ATTENTION !!!
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# ATTENTION !!!

#### 4. Добавляем в граф информацию из загрженного датасета

In [ ]:
rkg_main.update_memory(data)

#### 5. Q&A

In [ ]:
METRICS = ReaderMetrics(base_dir="../..", bs_model_path="google/electra-base-discriminator")
# METRICS.exact_match(gen_answers, trgt_answers)

In [1]:
# TODO

In [ ]:
examples_questions = []

In [ ]:
rkg_main.answer_question(examples_questions[0])